# Coverage-gap inspection - out-of-inventory forms

A **targeted coverage diagnostic**, not a second formal validation sample. It
exists to answer one question: of the DiMLex occurrences discopy never
enumerated as candidates, how many are genuine discourse relations?

30 cases drawn only from forms the deterministic triage marked plausible -
`given`, `given that`, `despite`, `eventually` - plus three `with` cases as a
contrast, since `with` is the largest out-of-inventory form and is overwhelmingly
non-connective.

**This does not touch the 50-case validation.** Different sample, different
answer file, different notebook. The completed 50-case results stay exactly as
they are.

**No accuracy metric comes out of this.** The sample is purposively selected
from one tail of a heuristic, so it supports counts and examples only - not a
rate, and certainly not recall.

In [1]:
# ============================================================
# Setup
# ============================================================

from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start=None, repo_name="masters_thesis_sdg"):
    current = (start or Path.cwd()).resolve()
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(
                f"Could not find repo root {repo_name!r} above {Path.cwd()}"
            )
        current = current.parent


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.sentences import split_sentences
from src.justification_analysis.validation.discourse_review_tools import (
    DiscourseReviewApp, ValidationStore, build_cases, render_case_html,
)
from src.justification_analysis.comparison.discourse_statistics import (
    load_justification_frame,
)

ARTIFACTS = (
    REPO_ROOT / "analysis" / "cross_model" / "base" / "voting"
    / "prompt_v4" / "justification_analysis" / "discourse_parser"
)

SAMPLE_PATH = ARTIFACTS / "coverage_inspection_sample_30.csv"
COMPLETED_PATH = ARTIFACTS / "coverage_inspection_completed.csv"

print("sample :", SAMPLE_PATH.name)
print("answers:", COMPLETED_PATH.name)

sample : coverage_inspection_sample_30.csv
answers: coverage_inspection_completed.csv


## The coverage gap being inspected

Context for what you are about to judge: the full triage over all 1,661
occurrences discopy never enumerated.

In [2]:
gap_summary = pd.read_csv(
    ARTIFACTS / "coverage_gap_summary_by_form.csv", encoding="utf-8-sig"
)
display(gap_summary)

plausible = int(gap_summary["CLAUSE_INITIAL_PLAUSIBLE"].sum())
total = int(gap_summary["total"].sum())
print(f"plausible: {plausible:,} / {total:,} ({100 * plausible / total:.1f}%)")
print("The triage is a syntactic heuristic, calibrated against the 50-case "
      "validation (8/10 agreement). This notebook checks the plausible tail.")

,marker,category,CLAUSE_INITIAL_PLAUSIBLE,NON_CONNECTIVE_LIKELY,UNCLEAR,total,pct_plausible
0,with,Contingency,62,661,111,834,7.4
1,given,Contingency,460,16,1,477,96.4
2,despite,Comparison,76,7,30,113,67.3
3,particularly,Expansion,0,74,0,74,0.0
4,eventually,Temporal,16,41,0,57,28.1
5,given that,Contingency,54,0,0,54,100.0
6,without,Contingency,5,31,0,36,13.9
7,upon,Temporal,1,11,0,12,8.3
8,in response to,Contingency,0,0,2,2,0.0
9,and,Expansion,0,0,1,1,0.0


plausible: 674 / 1,661 (40.6%)
The triage is a syntactic heuristic, calibrated against the 50-case validation (8/10 agreement). This notebook checks the plausible tail.


In [3]:
# ============================================================
# Load the 30 cases and any answers already given
# ============================================================

sample = pd.read_csv(SAMPLE_PATH, encoding="utf-8-sig")
justifications = load_justification_frame(REPO_ROOT)

cases = build_cases(sample, justifications, split_sentences)
store = ValidationStore(COMPLETED_PATH, cases)

assert len(cases) == 30, f"expected 30 cases, got {len(cases)}"
assert sample["validation_id"].is_unique
assert set(sample["failure_type"]) == {"not_enumerated"}, \
    "this sheet must contain only never-enumerated occurrences"

print(f"cases loaded    : {len(cases)}")
print(f"already answered: {store.n_answered()} / {len(cases)}")
display(sample.groupby(["marker", "triage"]).size()
        .rename("n").reset_index())

cases loaded    : 30
already answered: 30 / 30


,marker,triage,n
0,despite,CLAUSE_INITIAL_PLAUSIBLE,7
1,eventually,CLAUSE_INITIAL_PLAUSIBLE,4
2,given,CLAUSE_INITIAL_PLAUSIBLE,10
3,given that,CLAUSE_INITIAL_PLAUSIBLE,6
4,with,CLAUSE_INITIAL_PLAUSIBLE,3


## Review

For each case, judge:

- **Is this a valid discourse relation that discopy failed to report?**
- **If yes: which top-level PDTB category?**

discopy never proposed these spans, so there is no parser prediction to show -
only the DiMLex category and the triage label. The marker is highlighted from
its exact character offsets.

Every click saves to `coverage_inspection_completed.csv`.

In [4]:
app = DiscourseReviewApp(cases, store)
display(app.ui)

## Progress and summary

Re-run to see where you are. Once all 30 are answered this reports counts by
surface form - raw counts only, no rate.

In [5]:
store.save()

answers = store.to_frame()
answers["marker"] = [c["marker"] for c in cases]
answers["triage"] = [c.get("triage", "") for c in cases]

n_answered = store.n_answered()
print(f"answered: {n_answered} / {len(cases)}")
print(f"saved -> {COMPLETED_PATH}")

if n_answered < len(cases):
    remaining = [int(c["validation_id"]) for c in cases
                 if not store.is_answered(c)]
    print(f"\nstill to review: {remaining}")
else:
    valid = answers["manual_valid_relation_missed_by_discopy"].str.lower().eq("yes")
    print(f"\njudged valid missed relations: {int(valid.sum())}/{len(answers)}")
    print("\nby surface form:")
    display(
        answers.assign(valid=valid)
        .groupby("marker")
        .agg(n=("valid", "size"), judged_valid=("valid", "sum"))
        .assign(judged_invalid=lambda t: t["n"] - t["judged_valid"])
    )
    print("\nby predicted top-level category (where judged valid):")
    display(answers.loc[valid, "manual_top_level_category"]
            .value_counts().rename_axis("category").reset_index(name="n"))
    print("\nRaw counts only. The sample is purposively drawn from the "
          "plausible tail of a heuristic, so it supports counts and examples, "
          "not a rate and not recall.")

answered: 30 / 30
saved -> C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\cross_model\base\voting\prompt_v4\justification_analysis\discourse_parser\coverage_inspection_completed.csv

judged valid missed relations: 17/30

by surface form:


,n,judged_valid,judged_invalid
marker,,,
despite,7,1,6
eventually,4,0,4
given,10,10,0
given that,6,6,0
with,3,0,3



by predicted top-level category (where judged valid):


,category,n
0,Contingency,16
1,Comparison,1



Raw counts only. The sample is purposively drawn from the plausible tail of a heuristic, so it supports counts and examples, not a rate and not recall.


### Static view (optional)

Renders one case without recording anything.

In [6]:
from IPython.display import HTML, display as _display

CASE_TO_SHOW = 1   # 1-30

_display(HTML(render_case_html(
    cases[CASE_TO_SHOW - 1], CASE_TO_SHOW, len(cases)
)))

### What this feeds

The counts above inform the **A vs B** decision, specifically whether the
~674 plausible out-of-inventory occurrences represent enough genuine missed
relations to justify a DiMLex-candidate hybrid.

They do not settle it on their own. Even if these forms are largely genuine,
the separate question is whether discopy's sense classifier could label them:
`given`, `given that` and `despite` are outside its inventory because PDTB does
not treat them as explicit connectives, so they appeared in its training data
only as negatives. That would need its own probe.